# Oil Prices in 2026 — Act 5: What If the Model Could Read the News?

The [companion notebook](energy_oil_case_study.ipynb) showed that Prophet's rolling 30-day
forecast catastrophically missed the 2026 oil price surge — forecasting ~$61/bbl
while WTI hit $100. The model wasn't wrong in principle; it simply had no mechanism
for incorporating the geopolitical context that was publicly available at the time.

This notebook asks: **could a context-aware LLM forecaster have done better?**

We evaluate three key forecast origins in early 2026 using three methods side by side:
- **Prophet** (baseline — already computed, loaded from cache)
- **LLMP — no context** (Gemini 3 Flash, history only)
- **LLMP — with context** (same model + plausibly-knowable geopolitical context at each origin)

And we frame the comparison three ways:

| | Question type | Evaluation |
|---|---|---|
| **Act 5** | *Trajectory* — what will the 30-day price path look like? | MAE vs. actuals |
| **Act 6** | *Binary* — will price exceed a meaningful threshold in 30 days? | Calibration of P(exceed) |
| **Act 7** | *Causal* — what forces are the model anchoring on? | Qualitative reasoning audit |

In [13]:
from __future__ import annotations

import json
import logging
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.subplots as psp
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
logging.getLogger("prophet").setLevel(logging.ERROR)

# ── Repo root: walk up from CWD until pyproject.toml is found ─────────────────
_cwd = Path(os.getcwd()).resolve()
REPO_ROOT = _cwd
while not (REPO_ROOT / "pyproject.toml").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        REPO_ROOT = _cwd
        break
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"

for p in [str(REPO_ROOT / "implementations"), str(REPO_ROOT / "aieng-forecasting")]:
    if p not in sys.path:
        sys.path.insert(0, p)

load_dotenv(REPO_ROOT / ".env")

# ── Colour palette (matches companion notebook) ────────────────────────────────
CLR_HISTORY   = "#bdd7e7"
CLR_ACTUAL    = "#2171b5"   # solid blue — the truth
CLR_PROPHET   = "#636363"   # grey — the blind baseline
CLR_LLMP_BARE = "#fd8d3c"   # orange — LLM, history only
CLR_LLMP_CTX  = "#2ca02c"   # green — LLM, with context
CLR_CONFLICT  = "#d62728"   # red — conflict annotation
CONFLICT_DATE = pd.Timestamp("2026-03-01")

print(f"Repo root : {REPO_ROOT}")
print(f"Data dir  : {DATA_DIR}")
print("Setup complete.")

Repo root : /Users/ethanjackson/agentic-forecasting
Data dir  : /Users/ethanjackson/agentic-forecasting/data
Setup complete.


In [14]:
# ── WTI price history (same cache as companion notebook) ──────────────────────
PRICE_CACHE        = DATA_DIR / "wti_price_history.parquet"
PROPHET_CACHE      = DATA_DIR / "energy_case_study_forecasts_30d_daily_v3.parquet"
PROPHET_TRAJ_CACHE = DATA_DIR / "energy_prophet_trajectories.parquet"
LLMP_CACHE         = DATA_DIR / "energy_llmp_context_forecasts.parquet"

price_df = pd.read_parquet(PRICE_CACHE)
price_df.index = pd.DatetimeIndex(
    [pd.Timestamp(str(d)[:10]) for d in price_df.index]
)
price_df.index.name = "date"
price_df = price_df.sort_index()

prophet_df = pd.read_parquet(PROPHET_CACHE)
prophet_df["sim_day"]        = pd.to_datetime(prophet_df["sim_day"])
prophet_df["resolution_date"] = pd.to_datetime(prophet_df["resolution_date"])

print(f"WTI price history : {price_df.index[0].date()} → {price_df.index[-1].date()} ({len(price_df):,} days)")
print(f"Prophet forecasts : {prophet_df['sim_day'].min().date()} → {prophet_df['sim_day'].max().date()} ({len(prophet_df):,} rows)")

WTI price history : 2021-01-04 → 2026-05-01 (1,340 days)
Prophet forecasts : 2025-01-02 → 2026-04-01 (314 rows)


---

## The Setup

We pick **three forecast origins** in early 2026 — each representing a different
stage of the geopolitical escalation that drove WTI from ~$58 to $100+ between
January and April 2026.

| Origin | WTI at origin | Resolution date | Actual WTI at resolution | Prophet forecast | Context available |
|---|---|---|---|---|---|
| Jan 5, 2026 | $58 | Feb 4, 2026 | **$65** | $58 (inside CI) | Tensions building; OPEC+ cuts; insurance premiums rising |
| Feb 2, 2026 | $62 | Mar 4, 2026 | **$75** | $61 (miss — above CI) | Gulf of Oman incident; escalation fears; analyst upgrades |
| Mar 2, 2026 | $71 | Apr 1, 2026 | **$100** | $61 (catastrophic miss) | Conflict active; Strait of Hormuz blockade; IEA emergency session |

For each origin we ask: does an LLMP with access to publicly-available context
shift its forecast in the right direction — toward the actual outcome?

In [15]:
def compress_history(
    price_df: pd.DataFrame,
    as_of: pd.Timestamp,
    recent_window_months: int = 6,
) -> pd.DataFrame:
    """Return a token-efficient history: weekly averages for older data, daily for recent.

    Compresses ~1300 daily rows to ~200 rows while preserving the recent
    daily granularity that matters most for the LLM's short-horizon forecast.
    """
    hist = price_df[price_df.index <= as_of].copy()
    cutoff_daily = as_of - pd.DateOffset(months=recent_window_months)

    older = (
        hist[hist.index < cutoff_daily]
        .resample("W")
        .mean()
        .reset_index()
        .rename(columns={"date": "timestamp", "price": "value"})
    )
    recent = (
        hist[hist.index >= cutoff_daily]
        .reset_index()
        .rename(columns={"date": "timestamp", "price": "value"})
    )

    result = pd.concat([older, recent], ignore_index=True)
    result["timestamp"] = pd.to_datetime(result["timestamp"])
    return result[["timestamp", "value"]].sort_values("timestamp").reset_index(drop=True)


def prophet_row_at_origin(prophet_df: pd.DataFrame, origin: pd.Timestamp) -> pd.Series:
    """Return the Prophet forecast row whose sim_day is nearest to (on or after) origin."""
    candidates = prophet_df[prophet_df["sim_day"] >= origin]
    return candidates.iloc[0]


def resolution_price(price_df: pd.DataFrame, origin: pd.Timestamp, horizon_calendar_days: int = 30) -> tuple[pd.Timestamp, float]:
    """Return (resolution_date, actual_price) for a calendar-day horizon from origin."""
    target = origin + pd.Timedelta(days=horizon_calendar_days)
    row = price_df[price_df.index >= target].iloc[0]
    return row.name, float(row["price"])


# Sanity-check the three origins
ORIGINS = [
    pd.Timestamp("2026-01-05"),
    pd.Timestamp("2026-02-02"),
    pd.Timestamp("2026-03-02"),
]

# ── Shock experiment: 8 weekly origins spanning Feb–Mar 2026 ─────────────────
# 4 calm weeks followed by 4 shock weeks — a natural stress test.
SHOCK_ORIGINS = [
    pd.Timestamp("2026-02-02"),
    pd.Timestamp("2026-02-09"),
    pd.Timestamp("2026-02-17"),   # Tue (Mon = Presidents' Day holiday)
    pd.Timestamp("2026-02-23"),
    pd.Timestamp("2026-03-02"),
    pd.Timestamp("2026-03-09"),
    pd.Timestamp("2026-03-16"),
    pd.Timestamp("2026-03-23"),
]

print("Origin summary:")
for o in ORIGINS:
    price_at_origin = float(price_df[price_df.index >= o].iloc[0]["price"])
    res_date, res_price = resolution_price(price_df, o)
    p_row = prophet_row_at_origin(prophet_df, o)
    print(
        f"  {o.date()}  WTI=${price_at_origin:.2f}  "
        f"→ resolution {res_date.date()} actual=${res_price:.2f}  "
        f"prophet=${p_row['yhat']:.2f} [{p_row['yhat_lower']:.1f},{p_row['yhat_upper']:.1f}]  "
        f"inside_ci={p_row['inside_ci']}"
    )

Origin summary:
  2026-01-05  WTI=$58.32  → resolution 2026-02-04 actual=$65.14  prophet=$57.55 [49.5,65.5]  inside_ci=True
  2026-02-02  WTI=$62.14  → resolution 2026-03-04 actual=$74.66  prophet=$60.91 [52.8,69.4]  inside_ci=False
  2026-03-02  WTI=$71.23  → resolution 2026-04-01 actual=$100.12  prophet=$61.32 [53.3,69.3]  inside_ci=False


In [16]:
# ── Prophet full-trajectory forecasts for the three origins ──────────────────
#
# The existing Prophet cache stores only the terminal 30-calendar-day-ahead
# point per origin.  Here we re-run Prophet for just the three key origins and
# produce a 21-business-day trajectory fan — matching the LLMP output structure
# so the trajectory chart shows a like-for-like comparison between methods.
#
# Fitting 3 Prophet models takes ~10 s; results are cached to
# data/energy_prophet_trajectories.parquet so subsequent runs are instant.

from prophet import Prophet  # type: ignore[import-untyped]


def _fit_prophet_at_origin(price_df: pd.DataFrame, origin: pd.Timestamp) -> pd.DataFrame:
    """Fit one Prophet model on all data up to origin; return 21-business-day trajectory."""
    train_df = price_df.loc[:origin][["price"]].reset_index()
    train_df.columns = pd.Index(["ds", "y"])

    model = Prophet(
        interval_width=0.95,
        daily_seasonality=False,
        weekly_seasonality=False,
        yearly_seasonality=True,
        seasonality_mode="multiplicative",
    )
    model.fit(train_df)

    # Predict enough calendar days to cover 21 business days (≈ 31 calendar days)
    future = model.make_future_dataframe(periods=35, freq="D")
    pred   = model.predict(future).set_index("ds")

    bday_dates = pd.bdate_range(start=origin + pd.offsets.BDay(1), periods=21)
    rows = []
    for h, date in enumerate(bday_dates, start=1):
        cal_date = date.normalize()
        if cal_date in pred.index:
            row = pred.loc[cal_date]
        else:
            nearest_idx = int((pred.index - cal_date).abs().argmin())
            row = pred.iloc[nearest_idx]
        rows.append({
            "origin":        origin,
            "forecast_date": date,
            "horizon":       h,
            "yhat":          float(row["yhat"]),
            "yhat_lower":    float(row["yhat_lower"]),
            "yhat_upper":    float(row["yhat_upper"]),
        })

    return pd.DataFrame(rows)


def load_prophet_trajectories(
    price_df: pd.DataFrame,
    origins: list[pd.Timestamp],
    cache_path: Path,
) -> pd.DataFrame:
    """Load from cache or compute full Prophet trajectory for each origin."""
    if cache_path.exists():
        df = pd.read_parquet(cache_path)
        df["origin"]        = pd.to_datetime(df["origin"])
        df["forecast_date"] = pd.to_datetime(df["forecast_date"])
        print(f"Loaded {len(df)} Prophet trajectory rows from cache.")
        return df

    print("Fitting Prophet at 3 origins (~10 s)...")
    frames = []
    for origin in origins:
        print(f"  {origin.date()} ...", end=" ", flush=True)
        frames.append(_fit_prophet_at_origin(price_df, origin))
        print("done")

    df = pd.concat(frames, ignore_index=True)
    df.to_parquet(cache_path, index=False)
    print(f"Saved {len(df)} rows to {cache_path}")
    return df


prophet_traj_df = load_prophet_trajectories(price_df, ORIGINS, PROPHET_TRAJ_CACHE)
prophet_traj_df.head(6)

Loaded 63 Prophet trajectory rows from cache.


,origin,forecast_date,horizon,yhat,yhat_lower,yhat_upper
0,2026-01-05,2026-01-06,1,56.110597,47.564043,63.171764
1,2026-01-05,2026-01-07,2,56.226107,48.657983,64.782960
2,2026-01-05,2026-01-08,3,56.346479,48.503792,64.105882
3,2026-01-05,2026-01-09,4,56.470931,47.860042,64.401306
4,2026-01-05,2026-01-12,5,56.857824,48.616767,64.700857
5,2026-01-05,2026-01-13,6,56.986786,48.967157,65.217871


In [17]:
# ── Prophet trajectories for all 8 shock-experiment origins ──────────────────
# Reuse the same _fit_prophet_at_origin / load_prophet_trajectories helpers
# defined above, but with a separate cache so Act 5 data stays pristine.

PROPHET_SHOCK_TRAJ_CACHE = DATA_DIR / "energy_shock_prophet_trajectories.parquet"

prophet_shock_traj_df = load_prophet_trajectories(price_df, SHOCK_ORIGINS, PROPHET_SHOCK_TRAJ_CACHE)
print(
    f"Shock-origin Prophet trajectories: {len(prophet_shock_traj_df)} rows "
    f"across {prophet_shock_traj_df['origin'].nunique()} origins"
)
prophet_shock_traj_df.groupby("origin")["horizon"].agg(["min", "max", "count"])

Loaded 168 Prophet trajectory rows from cache.
Shock-origin Prophet trajectories: 168 rows across 8 origins


,min,max,count
origin,,,
2026-02-02,1,21,21
2026-02-09,1,21,21
2026-02-17,1,21,21
2026-02-23,1,21,21
2026-03-02,1,21,21
2026-03-09,1,21,21
2026-03-16,1,21,21
2026-03-23,1,21,21


In [18]:
# ── Context snippets — plausibly knowable on each origin date ─────────────────
# These represent the kind of information a professional energy analyst
# would have had access to from public sources: news, vessel-tracking
# services, futures data, and analyst reports published before the origin date.

ORIGIN_CONTEXTS: dict[str, dict] = {
    "2026-01-05": {
        "label": "Jan 5, 2026",
        "threshold_usd": 65.0,
        "context_text": (
            "As of January 5 2026:\n"
            "- WTI crude has been range-bound in the $56–62 band since October 2025 on soft "
            "demand signals and elevated US inventory builds.\n"
            "- OPEC+ is maintaining its current production-cut agreement through Q1 2026; "
            "no rollback has been signalled.\n"
            "- Iranian proxy forces conducted three separate attacks on US logistics assets "
            "in Iraq and Syria in Q4 2025. US-Iran tensions are elevated but have not "
            "escalated to direct military exchange.\n"
            "- Lloyd's of London hull-war insurance premiums for tankers transiting the "
            "Gulf of Oman have risen approximately 15% since September 2025.\n"
            "- The WTI NYMEX forward curve is in mild backwardation: front month $58, "
            "6-month forward approximately $56.\n"
            "- EIA weekly report (Dec 31 2025): US crude inventories 8% below the 5-year "
            "seasonal average."
        ),
    },
    "2026-02-02": {
        "label": "Feb 2, 2026",
        "threshold_usd": 72.0,
        "context_text": (
            "As of February 2 2026:\n"
            "- WTI gained approximately 7% in January, closing near $62, driven by "
            "escalating Persian Gulf tensions.\n"
            "- A US Navy escort mission in the Gulf of Oman was intercepted by Iranian "
            "fast-attack boats on January 28. No shots fired, but the incident was "
            "widely reported and prompted a diplomatic protest from Washington.\n"
            "- OPEC+ called an emergency ministerial consultation for February 10 amid "
            "concerns about supply-chain disruption risk; no production change announced yet.\n"
            "- Goldman Sachs revised its 2026 WTI price target upward to $70–85 in a "
            "February 1 research note, citing a 'geopolitical risk premium re-rating'.\n"
            "- Vessel-tracking data shows tanker transits through the Strait of Hormuz "
            "down approximately 15% week-over-week, as operators seek alternative routings.\n"
            "- Brent/WTI spread widened to $4.50, the largest since early 2024, as "
            "European buyers began bidding up non-Gulf grades.\n"
            "- US intelligence officials stated publicly that Iranian military assets "
            "have been repositioned closer to the Strait of Hormuz."
        ),
    },
    "2026-03-02": {
        "label": "Mar 2, 2026",
        "threshold_usd": 85.0,
        "context_text": (
            "As of March 2 2026:\n"
            "- The US conducted direct airstrikes on Iranian oil-infrastructure targets "
            "on March 1 2026 in response to an Iranian attack on a US carrier group "
            "in the Gulf of Oman on February 26.\n"
            "- Iran declared a partial blockade of the Strait of Hormuz effective "
            "March 1; approximately 20% of global seaborne oil supply transits the Strait.\n"
            "- WTI surged from $62 on February 2 to $71 by March 2 — a 14% move in "
            "one month — and front-month futures gapped up a further $4 at Monday open.\n"
            "- The IEA called an emergency ministerial meeting for March 5 to consider "
            "releasing strategic petroleum reserves.\n"
            "- Saudi Aramco issued force majeure declarations on several customer contracts; "
            "Saudi Arabia activated its emergency supply protocols.\n"
            "- Goldman Sachs issued an updated note on March 1 with a new 2026 WTI target "
            "of $95–115 and flagging a tail-risk scenario of $130 if the blockade persists "
            "beyond 60 days.\n"
            "- WTI NYMEX forward curve has swung into sharp backwardation: front month "
            "$71, 6-month forward $62, signalling market expectation of eventual resolution."
        ),
    },
}

print("Context snippets defined for:", list(ORIGIN_CONTEXTS.keys()))

Context snippets defined for: ['2026-01-05', '2026-02-02', '2026-03-02']


In [19]:
# ── Run LLMP forecasts (or load from cache) ───────────────────────────────────
#
# We use the LLMP module's internals directly so we can supply a pre-compressed
# history DataFrame rather than going through a DataService.  This is appropriate
# for a playground notebook; a production version would use a registered adapter.
#
# Each origin × 2 variants (bare / with-context) = 6 LLM calls.
# Results are cached to data/energy_llmp_context_forecasts.parquet.

from aieng.forecasting.methods.llm_processes.continuous import (
    ContinuousLLMPredictorConfig,
    _build_system_prompt,
    _build_user_prompt,
    _quantiles_per_step,
    _sample_trajectories,
    _stack_trajectories,
)
from aieng.forecasting.methods.llm_processes.base import serialize_history
from aieng.forecasting.evaluation.prediction import STANDARD_QUANTILES
from aieng.forecasting.evaluation.task import ForecastingTask
from aieng.forecasting.data.models import SeriesMetadata


MODEL       = "gemini/gemini-3-flash-preview"
N_SAMPLES   = 20
HORIZON_B   = 21   # ~21 business days ≈ 30 calendar days
PRECISION   = 2

_WTI_TASK = ForecastingTask(
    task_id="wti_crude_30d",
    target_series_id="wti_crude",
    horizons=list(range(1, HORIZON_B + 1)),
    frequency="B",
    description=(
        "WTI crude oil front-month futures price (USD/bbl), "
        "30 trading-day ahead probabilistic forecast. "
        "Forecast the daily closing price for each of the next "
        f"{HORIZON_B} business days."
    ),
)

_WTI_META = SeriesMetadata(
    series_id="wti_crude",
    description="WTI crude oil front-month futures (CL=F, Adj Close)",
    source="Yahoo Finance",
    units="USD/bbl",
    frequency="B",
)


def _run_one_forecast(
    price_df: pd.DataFrame,
    origin: pd.Timestamp,
    context_text: str | None,
    context_tag: str,
) -> dict:
    """Run LLMP at a single origin and return a dict of arrays."""
    history_df = compress_history(price_df, origin)
    history_str = serialize_history(history_df, precision=PRECISION)

    forecast_start = origin + pd.offsets.BDay(1)
    forecast_end   = origin + pd.offsets.BDay(HORIZON_B)

    system_prompt = _build_system_prompt()
    user_prompt   = _build_user_prompt(
        _WTI_TASK, history_str, _WTI_META,
        forecast_start, forecast_end, HORIZON_B,
        context_text=context_text,
    )

    cfg = ContinuousLLMPredictorConfig(
        model=MODEL,
        n_samples=N_SAMPLES,
        temperature=1.0,
        reasoning_effort="disable",
        context_text=context_text,
        context_tag=context_tag,
    )

    parsed, cost_usd, in_tok, out_tok, failures = _sample_trajectories(
        cfg=cfg, system_prompt=system_prompt, user_prompt=user_prompt
    )
    samples = _stack_trajectories([t.values for t in parsed], n_steps=HORIZON_B)
    q_grid  = _quantiles_per_step(samples)   # (HORIZON_B, len(STANDARD_QUANTILES))

    dates = pd.bdate_range(start=origin + pd.offsets.BDay(1), periods=HORIZON_B)

    rows = []
    for h_idx in range(HORIZON_B):
        row: dict = {
            "origin":      origin,
            "context_tag": context_tag,
            "forecast_date": dates[h_idx],
            "horizon":     h_idx + 1,
            "median":      float(q_grid[h_idx, STANDARD_QUANTILES.index(0.50)]),
            "cost_usd":    cost_usd,
        }
        for qi, q in enumerate(STANDARD_QUANTILES):
            row[f"q{int(q * 100):02d}"] = float(q_grid[h_idx, qi])
        rows.append(row)

    print(
        f"    origin={origin.date()} tag={context_tag:12s} "
        f"cost=${cost_usd:.4f}  failures={failures}/{N_SAMPLES}"
    )
    return {"rows": rows, "samples": samples.tolist()}


def run_all_forecasts(price_df: pd.DataFrame, cache_path: Path) -> pd.DataFrame:
    """Run (or load) all 6 LLMP forecasts; return a single flat DataFrame."""
    if cache_path.exists():
        df = pd.read_parquet(cache_path)
        df["origin"]        = pd.to_datetime(df["origin"])
        df["forecast_date"] = pd.to_datetime(df["forecast_date"])
        print(f"Loaded {len(df)} LLMP forecast rows from cache.")
        return df

    all_rows: list[dict] = []
    print("Running LLMP forecasts (6 API calls)...")
    for origin in ORIGINS:
        key = origin.strftime("%Y-%m-%d")
        ctx = ORIGIN_CONTEXTS[key]
        for tag, text in [("bare", None), ("context", ctx["context_text"])]:
            result = _run_one_forecast(price_df, origin, text, tag)
            all_rows.extend(result["rows"])

    df = pd.DataFrame(all_rows)
    df.to_parquet(cache_path, index=False)
    print(f"Saved {len(df)} rows to {cache_path}")
    return df


llmp_df = run_all_forecasts(price_df, LLMP_CACHE)
print(f"\nOrigins: {sorted(llmp_df['origin'].dt.date.unique())}")
print(f"Tags:    {sorted(llmp_df['context_tag'].unique())}")
llmp_df.head(6)

Loaded 126 LLMP forecast rows from cache.

Origins: [datetime.date(2026, 1, 5), datetime.date(2026, 2, 2), datetime.date(2026, 3, 2)]
Tags:    ['bare', 'context']


,origin,context_tag,forecast_date,horizon,median,cost_usd,q05,q10,q20,q30,q40,q50,q60,q70,q80,q90,q95
0,2026-01-05,bare,2026-01-06,1,58.450,0.07594,58.1500,58.420,58.450,58.450,58.450,58.450,58.450,58.450,58.624,58.650,58.6595
1,2026-01-05,bare,2026-01-07,2,58.820,0.07594,58.3955,58.410,58.620,58.720,58.780,58.820,58.820,58.820,59.120,59.123,59.1530
2,2026-01-05,bare,2026-01-08,3,59.080,0.07594,58.1935,58.300,58.590,58.769,58.846,59.080,59.150,59.150,59.150,59.180,59.4515
3,2026-01-05,bare,2026-01-09,4,58.835,0.07594,58.0910,58.120,58.394,58.700,58.724,58.835,58.874,59.158,59.226,59.384,59.4315
4,2026-01-05,bare,2026-01-12,5,59.165,0.07594,57.8850,57.937,58.426,58.550,59.052,59.165,59.232,59.323,59.420,59.787,59.8515
5,2026-01-05,bare,2026-01-13,6,59.280,0.07594,57.9150,58.400,58.686,58.959,59.092,59.280,59.368,59.680,59.904,60.120,60.1200


---

## Act 5 — Trajectory: Can the Model See the Move Coming?

Each panel below shows the 30-day forecast fan from one origin date.
Three forecast traces:
- **Grey** — Prophet (statistical baseline; history only)
- **Orange** — LLMP, history only (same information as Prophet, different model family)
- **Green** — LLMP with context (public geopolitical information available on that date)

The **solid blue line** is the realized WTI price (actual outcome).
Shading shows 50% and 90% credible intervals for the LLMP forecasts.

In [20]:
def _add_fan(
    fig: go.Figure,
    dates: "pd.Series",
    lower: "pd.Series",
    upper: "pd.Series",
    median: "pd.Series",
    color: str,
    opacity_band: float,
    name: str,
    show_legend: bool,
    legendgroup: str,
    row: int,
    col: int,
) -> None:
    """Add a forecast fan (CI band + median line) to a subplot panel."""
    # CI shading (fill between lower and upper)
    fig.add_trace(
        go.Scatter(
            x=pd.concat([dates, dates[::-1]]),
            y=pd.concat([lower, upper[::-1]]),
            fill="toself",
            fillcolor=color,
            opacity=opacity_band,
            line=dict(width=0),
            mode="lines",
            showlegend=False,
        ),
        row=row, col=col,
    )
    # Median line
    fig.add_trace(
        go.Scatter(
            x=dates, y=median,
            line=dict(color=color, width=2),
            name=name if show_legend else None,
            showlegend=show_legend,
            legendgroup=legendgroup,
        ),
        row=row, col=col,
    )


def make_trajectory_figure(
    price_df: pd.DataFrame,
    prophet_traj_df: pd.DataFrame,
    llmp_df: pd.DataFrame,
    origins: list[pd.Timestamp],
) -> go.Figure:
    """Three-column subplot: one panel per origin, like-for-like trajectory fan comparison.

    All three methods (Prophet, LLMP-bare, LLMP-with-context) are shown as a
    CI-band + median line over the full 21-business-day forecast horizon, making
    the comparison visually consistent.
    """
    labels = [ORIGIN_CONTEXTS[o.strftime("%Y-%m-%d")]["label"] for o in origins]

    fig = psp.make_subplots(
        rows=1, cols=3,
        subplot_titles=labels,
        shared_yaxes=False,
        horizontal_spacing=0.06,
    )

    for col, origin in enumerate(origins, start=1):
        # ── History (60 days pre-origin) ──────────────────────────────────────
        hist_start = origin - pd.Timedelta(days=60)
        hist = price_df[price_df.index >= hist_start].loc[:origin]
        fig.add_trace(
            go.Scatter(
                x=hist.index, y=hist["price"],
                line=dict(color=CLR_HISTORY, width=2),
                name="History" if col == 1 else None,
                showlegend=(col == 1),
                legendgroup="history",
            ),
            row=1, col=col,
        )

        # ── Actuals post-origin ────────────────────────────────────────────────
        res_date, _ = resolution_price(price_df, origin)
        actuals = price_df[(price_df.index > origin) & (price_df.index <= res_date)]
        fig.add_trace(
            go.Scatter(
                x=actuals.index, y=actuals["price"],
                line=dict(color=CLR_ACTUAL, width=2.5),
                name="Actual" if col == 1 else None,
                showlegend=(col == 1),
                legendgroup="actual",
            ),
            row=1, col=col,
        )

        # ── Prophet trajectory fan ─────────────────────────────────────────────
        pt = prophet_traj_df[prophet_traj_df["origin"] == origin].sort_values("forecast_date")
        if not pt.empty:
            _add_fan(
                fig,
                dates=pt["forecast_date"],
                lower=pt["yhat_lower"],
                upper=pt["yhat_upper"],
                median=pt["yhat"],
                color=CLR_PROPHET,
                opacity_band=0.15,
                name="Prophet 95% CI",
                show_legend=(col == 1),
                legendgroup="prophet",
                row=1, col=col,
            )

        # ── LLMP fans (bare + context) ─────────────────────────────────────────
        for tag, clr, name in [
            ("bare",    CLR_LLMP_BARE, "LLMP — history only"),
            ("context", CLR_LLMP_CTX,  "LLMP — with context"),
        ]:
            sub = llmp_df[(llmp_df["origin"] == origin) & (llmp_df["context_tag"] == tag)].sort_values("forecast_date")
            if sub.empty:
                continue

            # 90% CI (outer, light)
            _add_fan(
                fig,
                dates=sub["forecast_date"],
                lower=sub["q05"],
                upper=sub["q95"],
                median=sub["median"] if "median" in sub.columns else sub["q50"],
                color=clr,
                opacity_band=0.10,
                name=name,
                show_legend=(col == 1),
                legendgroup=f"llmp_{tag}",
                row=1, col=col,
            )
            # 50% CI (inner, darker)
            if "q25" in sub.columns and "q75" in sub.columns:
                fig.add_trace(
                    go.Scatter(
                        x=pd.concat([sub["forecast_date"], sub["forecast_date"][::-1]]),
                        y=pd.concat([sub["q25"], sub["q75"][::-1]]),
                        fill="toself",
                        fillcolor=clr,
                        opacity=0.20,
                        line=dict(width=0),
                        mode="lines",
                        showlegend=False,
                    ),
                    row=1, col=col,
                )

        # ── Origin marker ──────────────────────────────────────────────────────
        fig.add_vline(
            x=origin.timestamp() * 1000,
            line=dict(color="#888", dash="dash", width=1),
            row=1, col=col,
        )

    fig.update_layout(
        title=dict(
            text="30-Day WTI Forecast Trajectories — Prophet vs. LLMP vs. LLMP + Context",
            font=dict(size=15),
        ),
        height=420,
        width=1200,
        legend=dict(orientation="h", y=-0.18),
        template="plotly_white",
        margin=dict(t=60, b=90),
    )
    fig.update_yaxes(title_text="WTI (USD/bbl)", col=1)
    return fig


make_trajectory_figure(price_df, prophet_traj_df, llmp_df, ORIGINS).show()

The January origin stays within Prophet's CI. February and March do not — those are the
cases where context makes the difference. Prophet's 30-day median ends up ~$61 for all three
regardless of what's happening in the world. The LLMP with context shifts upward when the
news warrants it.

---

## Act 6 — Binary Forecasting: Will WTI Rise by More Than $5 This Week?

Act 5 showed that trajectory forecasts diverge sharply once context enters the picture.
Here we pin it to a single, evaluable binary question:

> **P(WTI day-5 close > today's price + $5/bbl)** — 8 weekly origins, Feb 2 – Mar 23, 2026

**Why Prophet fails here structurally:** Prophet always forecasts reversion toward its
long-run trend (~$63). When WTI is at $71–$95, Prophet's implied probability that prices
rise *further* by $5+ is essentially zero. But they did — by $23 on Mar 2 and $15 on Mar 23.
The Analyst Agent reads the news and disagrees.

| Origin | WTI | 5-day Δ | Outcome |
|---|---|---|---|
| Feb 2 | $62.14 | +$2.22 | No |
| Feb 9 | $64.36 | −$2.03 | No |
| Feb 17 | $62.33 | +$3.30 | No |
| Feb 23 | $66.31 | +$4.92 | No |
| Mar 2 | $71.23 | **+$23.54** | **Yes — upward shock** |
| Mar 9 | $94.77 | −$1.27 | No |
| Mar 16 | $93.50 | −$5.37 | No |
| Mar 23 | $88.13 | **+$14.75** | **Yes — upward shock** |

In [21]:
import asyncio
import json
import re

import scipy.interpolate
import scipy.stats
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.genai import types as genai_types
from google.genai.types import GenerateContentConfig
import litellm

# ── Parameters ────────────────────────────────────────────────────────────────
SHOCK_THRESHOLD      = 5.0   # day-5 close > today + $5 = "upward shock"
SHOCK_HORIZON        = 5     # business days (1 trading week)
SHOCK_ANALYST_CACHE  = DATA_DIR / "energy_upshock_analyst_forecasts.json"
SHOCK_CONTEXT_CACHE  = DATA_DIR / "energy_upshock_news_context.json"

# ── Ground truth ──────────────────────────────────────────────────────────────
def check_shock_outcome(
    price_df: pd.DataFrame,
    origin: pd.Timestamp,
    threshold: float,
    horizon_bdays: int,
) -> tuple[int, float]:
    """Return (outcome, delta): outcome=1 if day-H close > origin_price + threshold."""
    origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])
    future_days  = price_df[price_df.index > origin].iloc[:horizon_bdays]
    delta        = float(future_days.iloc[-1]["price"]) - origin_price
    return (1 if delta > threshold else 0), delta

# ── Prophet P(upward shock) ───────────────────────────────────────────────────
def prophet_prob_shock(
    prophet_traj_sub: pd.DataFrame,
    origin_price: float,
    threshold: float,
    horizon: int = SHOCK_HORIZON,
) -> float:
    """P(price_h > origin + threshold) from Prophet's 95% CI (Gaussian approx)."""
    row = prophet_traj_sub[prophet_traj_sub["horizon"] == horizon]
    if row.empty:
        return float("nan")
    row = row.iloc[0]
    sigma = (float(row["yhat_upper"]) - float(row["yhat_lower"])) / (2 * 1.96)
    if sigma <= 0:
        return 1.0 if float(row["yhat"]) > origin_price + threshold else 0.0
    return float(np.clip(
        1.0 - scipy.stats.norm.cdf(origin_price + threshold, loc=float(row["yhat"]), scale=sigma),
        0.0, 1.0,
    ))

# ── Context Agent (Google ADK + google_search) ────────────────────────────────
_CONTEXT_SYSTEM = """\
You are an oil market intelligence specialist with access to web search.

CRITICAL TEMPORAL CONSTRAINT — you are simulating the perspective of an analyst
as of {cutoff_date}.
- Include ONLY information that was publicly available BEFORE {cutoff_date}.
- EXCLUDE any events, market moves, or data from {cutoff_date} or later.
- If a search result appears to post-date the cutoff, skip it entirely.
- This constraint is absolute. Violating it would corrupt the forecast.

Search for and summarise oil-market-relevant information focused on:
- WTI/Brent crude price level and recent trend
- OPEC+ production decisions and supply outlook
- Geopolitical risks in the Persian Gulf, Middle East, key shipping lanes
- US Strategic Petroleum Reserve and energy policy signals
- Notable tanker/shipping incidents or supply chain disruption signals
- Any published analyst forecasts or unusual price-target revisions

Focus especially on factors that could cause a *sudden large move* in WTI crude
over the next 5–10 days. Return a concise structured markdown summary (3–5 paragraphs).\
"""

async def _retrieve_oil_context_async(cutoff_date: str) -> str:
    """Run the Context Agent and return its oil-market summary as of cutoff_date."""
    _APP = "oil-context-agent"
    session_svc = InMemorySessionService()
    agent = LlmAgent(
        name="oil_context_agent",
        instruction=_CONTEXT_SYSTEM.format(cutoff_date=cutoff_date),
        tools=[google_search],
        model="gemini-3-flash-preview",
        generate_content_config=GenerateContentConfig(temperature=0.1, max_output_tokens=2048),
    )
    runner = Runner(agent=agent, app_name=_APP, session_service=session_svc)
    session = await session_svc.create_session(app_name=_APP, user_id="nb")
    prompt = (
        f"Provide an oil market intelligence briefing as of {cutoff_date}. "
        f"Focus on supply risks, OPEC+ policy, Persian Gulf geopolitics, and any factors "
        f"that could cause a sudden large price move in WTI crude in the next 5–10 days. "
        f"IMPORTANT: only use information available before {cutoff_date}."
    )
    content = genai_types.Content(role="user", parts=[genai_types.Part(text=prompt)])
    async for event in runner.run_async(user_id="nb", session_id=session.id, new_message=content):
        if event.is_final_response() and event.content and event.content.parts:
            return event.content.parts[0].text or ""
    return ""

# ── Analyst Agent (litellm / Gemini Flash) ────────────────────────────────────
_ANALYST_SYSTEM = """\
You are an expert oil market analyst making short-term probabilistic forecasts.

You will receive:
  1. Recent WTI crude oil price history (daily, most recent last)
  2. An oil market intelligence briefing with a strict temporal cutoff

Your task: estimate P(up) — the probability that WTI will close MORE THAN
${threshold}/bbl HIGHER than today's price at the end of {horizon} trading days.

This is a directional upside question only.

Be calibrated:
- No unusual upside catalyst → base rate ~10-15%.
- Escalating but unconfirmed geopolitical risk → 20-40%.
- Confirmed major supply disruption actively driving prices higher → 60-85%.

Respond ONLY in valid JSON with exactly these fields:
{{
  "probability_up": <float between 0 and 1>,
  "direction_bias": "<up|down|neutral>",
  "reasoning": "<2-4 sentences>",
  "key_signals": ["<signal 1>", "<signal 2>", "<signal 3>"],
  "confidence": "<high|medium|low>"
}}
Output ONLY the JSON object. No other text.\
"""

def _run_analyst(
    history_str: str,
    news_context: str,
    origin_date: str,
    origin_price: float,
) -> dict:
    """Analyst Agent: reason from price history + news context → structured dict."""
    system = _ANALYST_SYSTEM.format(horizon=SHOCK_HORIZON, threshold=int(SHOCK_THRESHOLD))
    target = origin_price + SHOCK_THRESHOLD
    user_prompt = (
        f"### WTI Price History (ending {origin_date})\n\n"
        f"{history_str}\n\n"
        f"---\n"
        f"### Oil Market Briefing (as of {origin_date})\n\n"
        f"{news_context}\n\n"
        f"---\n"
        f"Current WTI price: ${origin_price:.2f}/bbl on {origin_date}.\n"
        f"Estimate P(WTI closes above ${target:.2f} in {SHOCK_HORIZON} trading days)."
    )
    response = litellm.completion(
        model="gemini/gemini-3-flash-preview",
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user_prompt}],
        temperature=0.2,
        response_format={"type": "json_object"},
    )
    raw = response.choices[0].message.content or "{}"
    raw = re.sub(r"^```(?:json)?\s*", "", raw.strip())
    raw = re.sub(r"\s*```$", "", raw)
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r'"probability_up"\s*:\s*([\d.]+)', raw)
        prob = float(m.group(1)) if m else float("nan")
        return {"reasoning": raw[:500], "probability_up": prob,
                "key_signals": [], "direction_bias": "?", "confidence": "?"}

print("Agent definitions loaded.")
print(f"  SHOCK_THRESHOLD : +${SHOCK_THRESHOLD:.0f}/bbl  |  SHOCK_HORIZON : {SHOCK_HORIZON} bdays")
print(f"  Question        : P(WTI day-5 close > today + ${SHOCK_THRESHOLD:.0f})")
print(f"  Context Agent   : Google ADK + google_search  (gemini-3-flash-preview, temporal cutoff)")
print(f"  Analyst Agent   : gemini-3-flash-preview  →  JSON  (probability_up + reasoning)")

Agent definitions loaded.
  SHOCK_THRESHOLD : $5/bbl  |  SHOCK_HORIZON : 5 business days
  Context Agent   : Google ADK + google_search  (temporal cutoff enforced, gemini-2.0-flash)
  Analyst Agent   : gemini-2.5-flash  →  JSON  (P(shock) + reasoning chain)


In [22]:
# ── Run pipeline (or load from cache) ────────────────────────────────────────
# For each of 8 weekly shock-experiment origins:
#   (1) Context Agent fetches news; (2) Analyst Agent estimates P(shock).
# Top-level `await` works in Jupyter — uses the kernel's running event loop.

if SHOCK_ANALYST_CACHE.exists() and SHOCK_CONTEXT_CACHE.exists():
    with open(SHOCK_ANALYST_CACHE) as f:
        shock_analyst_results: list[dict] = json.load(f)
    with open(SHOCK_CONTEXT_CACHE) as f:
        shock_news_contexts: dict[str, str] = json.load(f)
    print(f"Loaded {len(shock_analyst_results)} cached shock-experiment forecasts.")
else:
    shock_news_contexts = {}
    shock_analyst_results = []

    for origin in SHOCK_ORIGINS:
        key          = origin.strftime("%Y-%m-%d")
        cutoff       = (origin - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
        origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])

        print(f"\n{'─' * 60}")
        print(f"Origin {key}  (context cutoff: {cutoff})  WTI ${origin_price:.2f}")

        print("  [1/2] Context Agent — searching for oil market intelligence ...")
        ctx = await _retrieve_oil_context_async(cutoff)
        shock_news_contexts[key] = ctx
        print(f"        {len(ctx):,} chars retrieved")

        print(f"  [2/2] Analyst Agent — reasoning about P(shock > ${SHOCK_THRESHOLD:.0f}) ...")
        hist     = compress_history(price_df, origin)
        hist_str = serialize_history(hist, precision=2)
        result   = _run_analyst(hist_str, ctx, key, origin_price)
        result["origin"] = key
        shock_analyst_results.append(result)
        p_up = result.get("probability_up", float("nan"))
        p_str = f"{p_up:.2f}" if isinstance(p_up, float) else str(p_up)
        print(f"        P(up>+${SHOCK_THRESHOLD:.0f})={p_str}  confidence={result.get('confidence', '?')}")

    with open(SHOCK_ANALYST_CACHE, "w") as f:
        json.dump(shock_analyst_results, f, indent=2)
    with open(SHOCK_CONTEXT_CACHE, "w") as f:
        json.dump(shock_news_contexts, f, indent=2)
    print("\nSaved to cache.")

# ── Assemble binary_df: Prophet + Analyst Agent + Always-50% baseline ─────────
shock_analyst_by_origin = {r["origin"]: r for r in shock_analyst_results}

binary_rows = []
for origin in SHOCK_ORIGINS:
    key          = origin.strftime("%Y-%m-%d")
    origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])
    outcome, max_move = check_shock_outcome(price_df, origin, SHOCK_THRESHOLD, SHOCK_HORIZON)

    # Prophet: P(shock) from Gaussian approximation to 5-day trajectory CI
    pt_sub = prophet_shock_traj_df[prophet_shock_traj_df["origin"] == origin]
    p_prob = prophet_prob_shock(pt_sub, origin_price, SHOCK_THRESHOLD)

    # Analyst Agent
    a      = shock_analyst_by_origin.get(key, {})
    a_prob = float(a.get("probability_up", float("nan")))

    for method, prob in [("Prophet", p_prob), ("Analyst Agent", a_prob), ("Always 50%", 0.5)]:
        binary_rows.append({
            "origin": key,
            "origin_price": origin_price,
            "max_move": max_move,
            "outcome": outcome,
            "method": method,
            "prob": prob,
            "brier": (prob - outcome) ** 2,
            "reasoning": a.get("reasoning") if method == "Analyst Agent" else None,
            "key_signals": a.get("key_signals", []) if method == "Analyst Agent" else [],
            "confidence": a.get("confidence") if method == "Analyst Agent" else None,
            "direction_bias": a.get("direction_bias") if method == "Analyst Agent" else None,
        })

binary_df = pd.DataFrame(binary_rows)
binary_df["origin_dt"] = pd.to_datetime(binary_df["origin"])
binary_df = binary_df.sort_values(["origin_dt", "method"]).reset_index(drop=True)

# ── Quick sanity check ─────────────────────────────────────────────────────────
print("\n── Binary forecast summary ─────────────────────────────────────────────")
summary = binary_df.pivot_table(
    index="origin", columns="method", values=["prob", "brier"], aggfunc="first"
)
print(summary.to_string())

Loaded 8 cached shock-experiment forecasts.

── Binary forecast summary ─────────────────────────────────────────────
                brier                                   prob                        
method     Always 50% Analyst Agent       Prophet Always 50% Analyst Agent   Prophet
origin                                                                              
2026-02-02       0.25        0.1764  1.414233e-01        0.5          0.42  0.376063
2026-02-09       0.25        0.0484  2.163626e-01        0.5          0.22  0.465148
2026-02-17       0.25        0.1764  5.412171e-02        0.5          0.42  0.232641
2026-02-23       0.25        0.1764  1.438595e-01        0.5          0.42  0.379288
2026-03-02       0.25        0.0025  7.531648e-02        0.5          0.95  0.725562
2026-03-09       0.25        0.0144  3.053219e-15        0.5          0.88  1.000000
2026-03-16       0.25        0.0225  4.661420e-11        0.5          0.85  0.999993
2026-03-23       0.25        0.0

In [23]:
# ── Display: news context + agent reasoning for all 8 shock origins ───────────
from IPython.display import display, Markdown as MD

for origin in SHOCK_ORIGINS:
    key          = origin.strftime("%Y-%m-%d")
    a            = shock_analyst_by_origin.get(key, {})
    ctx          = shock_news_contexts.get(key, "")
    origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])
    outcome, max_move = check_shock_outcome(price_df, origin, SHOCK_THRESHOLD, SHOCK_HORIZON)

    outcome_icon = "🔴 SHOCK" if outcome else "🟢 Calm"
    a_prob = float(a.get("probability_up", float("nan")))
    bs     = (a_prob - outcome) ** 2
    delta_str = f"{mark:.2f}" if (mark := float(price_df[price_df.index > origin].iloc[:SHOCK_HORIZON].iloc[-1]["price"]) - origin_price) >= 0 else f"{mark:.2f}"

    md = (
        f"---\n"
        f"### {key} — WTI ${origin_price:.2f}/bbl  ·  {outcome_icon}"
        f"  (5-day Δ: {delta_str:+.2f}, threshold +${SHOCK_THRESHOLD:.0f})\n\n"
        f"<details><summary>📰 Oil market context (cutoff: "
        f"{(origin - pd.Timedelta(days=1)).strftime('%Y-%m-%d')}) — click to expand</summary>\n\n"
        f"{ctx}\n\n</details>\n\n"
        f"**Reasoning:** {a.get('reasoning', 'N/A')}  \n"
        f"**Key signals:** {', '.join(a.get('key_signals', []))}  \n"
        f"**Bias:** {a.get('direction_bias', '?')}  "
        f"**Confidence:** {a.get('confidence', '?')}  "
        f"**P(up > +${SHOCK_THRESHOLD:.0f}) = {a_prob:.0%}**  "
        f"**Brier:** {bs:.3f}"
    )
    display(MD(md))

---
### 2026-02-02 — WTI $62.14/bbl  ·  🟢 Calm  (max 5-day move: $3.00, threshold $5)

<details><summary>📰 Oil market context (cutoff: 2026-02-01) — click to expand</summary>

This oil market intelligence briefing is prepared as of **February 1, 2026**, based on market conditions and data available up to this date.

### **Price Levels and Market Trend**
As of early February 2026, WTI crude is trading in a volatile range between **$58 and $64 per barrel**, while Brent maintains a premium of approximately $4–$5. The broad trend over the final quarter of 2025 was bearish, driven by a persistent global inventory build as non-OPEC+ production—led by the U.S., Brazil, and Guyana—consistently outpaced stalling demand growth in China and Europe. However, prices have found a firm floor in recent weeks following OPEC+’s decision to halt planned production increases. The market is currently characterized by a "tug-of-war" between bearish fundamental oversupply and a stubborn geopolitical risk premium that remains embedded in the forward curve.

### **OPEC+ Production Decisions and Supply Outlook**
In its most recent ministerial monitoring session (late 2025), OPEC+ reaffirmed its commitment to market stability by **pausing the scheduled unwinding of voluntary production cuts** for the first quarter of 2026. The group is currently keeping approximately 3.6 million bpd off the market through the end of the year. A critical focus for the next 5–10 days is the compliance of "over-producers"—specifically Iraq, Kazakhstan, and Russia—who are under intense pressure to implement "compensation cuts" through June 2026. While U.S. production remains near record highs of 13.6 million bpd, recent rig count declines suggest that domestic growth is finally plateauing, potentially tightening the medium-term supply outlook if OPEC+ maintains its current discipline.

### **Geopolitical Risks and Shipping Lanes**
Geopolitical tension remains the primary catalyst for a "sudden large move." While the **Red Sea** saw a fragile ceasefire in early 2025, shipping through the Bab el-Mandeb remains restricted, with a risk premium of $2–$4 per barrel still priced in due to sporadic Houthi activity and high insurance surcharges. More critically, the **Persian Gulf** has entered a period of "persistent exigency." Following the U.S. administration’s renewed "maximum pressure" rhetoric and recent Israeli strikes on regional proxies in late 2025, the threat of Iranian retaliation against tanker traffic in the **Strait of Hormuz** has reached its highest level in years. Any localized skirmish or "shadow war" escalation in these waters would likely trigger an immediate $10+ spike in WTI.

### **Supply Disruption Signals and 5-10 Day Outlook**
Over the next 5–10 days, the market is hyper-sensitive to two main factors:
1.  **Escalation in the Persian Gulf:** Intelligence reports

</details>

**Analyst Agent reasoning:**

> The market is currently in a state of 'persistent exigency' with the threat of Iranian retaliation in the Strait of Hormuz at its highest level in years, creating a significant geopolitical tail risk. While fundamentals suggest a range-bound market, the hyper-sensitivity to potential supply disruptions in the Persian Gulf makes a $5 price spike more likely than a move based on supply-demand balances alone. A $5 move represents approximately an 8% shift from current levels, which is consistent with historical reactions to localized skirmishes in key shipping lanes.

**Key signals:** Strait of Hormuz threat levels, OPEC+ compliance from Iraq and Russia, U.S. shale production plateauing  
**Direction bias:** up  **Confidence:** medium  
**P(shock > $5) = 42%**  **Brier score:** 0.176

---
### 2026-02-09 — WTI $64.36/bbl  ·  🟢 Calm  (max 5-day move: $2.03, threshold $5)

<details><summary>📰 Oil market context (cutoff: 2026-02-08) — click to expand</summary>

As of February 8, 2026, the crude oil market is characterized by a fragile stability following a volatile 2025. WTI crude is currently trading in the **$63–$66 range**, having recovered from a five-month low of approximately $60 reached in late October 2025. This recent support has been driven

</details>

**Analyst Agent reasoning:**

> WTI is currently consolidating in the $63-$66 range following a period of 'fragile stability.' While late January saw a near-$5 swing, current price action suggests a lack of immediate catalysts to drive a 7.8% move within a single week.

**Key signals:** Consolidation in the $63-$66 range, Absence of immediate geopolitical supply disruptions, Recent recovery from late 2025 lows  
**Direction bias:** neutral  **Confidence:** medium  
**P(shock > $5) = 22%**  **Brier score:** 0.048

---
### 2026-02-17 — WTI $62.33/bbl  ·  🟢 Calm  (max 5-day move: $4.10, threshold $5)

<details><summary>📰 Oil market context (cutoff: 2026-02-16) — click to expand</summary>

As of February 16, 2026, the crude oil market is characterized by a fragile "strategic equilibrium," with prices currently range-bound but heavily supported by a mounting geopolitical risk premium. WTI is trading between **$62 and $66 per barrel**, while Brent remains in the **$66–$70** range. Despite a bearish fundamental outlook for 2026—driven by projected global inventory builds of nearly 2 million bpd—prices have trended upward since January as markets price in a $4–$10 "wild card" premium due to escalating tensions in the Middle East.

**OPEC+ Policy and Supply Outlook**
The OPEC+ "eight countries" (led by Saudi Arabia and Russia) reaffirmed on February 1, 2026, their decision to pause all production increases through March. This extension of the Q1 production freeze is a response to seasonal demand weakness and a cautious effort to maintain price stability near the $65–$70 Brent floor. While the group has signaled a desire to gradually return 1.65 million bpd to the market starting in April, this remains strictly contingent on "evolving market conditions." Compliance remains high, with January production hitting a five-month low of 28.83 million bpd, signaling the group's commitment to preventing an early-year glut.

**Geopolitical Risks and Shipping Lanes**
The primary driver of potential volatility is the deteriorating US-Iran standoff following failed nuclear negotiations in Geneva and a brief but intense 12-day air conflict in late 2025. Intelligence reports indicate that Iran has significantly increased its oil export rates this week (Feb 15–20) to clear out storage, a move analysts interpret as a defensive measure against potential strikes on energy infrastructure. While Red Sea shipping had seen a fragile recovery in early 2026 following the October 2025 ceasefire, Houthi threats remain a persistent "tail risk" that could once again force major carriers to divert around the Cape of Good Hope.

**Factors for a Sudden Large Move (Next 5–10 Days)**
The market is currently hyper-sensitive to any signal of military escalation in the Persian Gulf. Analysts warn that a direct conflict or a blockade of the **Strait of Hormuz**—which handles 20% of global seaborne oil—would cause an immediate and massive price spike. Given that Brent is already trading roughly $10 above its "fair value" in anticipation of US military action, any kinetic event in the next 10 days could trigger a "gap up" in WTI toward the $80–$90 range. Conversely, any surprise diplomatic breakthrough or a signal from OPEC+ that they will accelerate production hikes to offset regional risk would likely cause a sharp correction toward the $55–$58 level.

</details>

**Analyst Agent reasoning:**

> The market is currently in a state of 'hyper-sensitivity' due to the US-Iran standoff and potential threats to the Strait of Hormuz, which handles 20% of global supply. While fundamentals are bearish for 2026, the immediate 5-day window is dominated by a $4-$10 geopolitical risk premium that could rapidly expand or collapse based on military developments. The current Iranian effort to clear storage suggests a defensive posture that often precedes regional escalation.

**Key signals:** US-Iran military posturing in the Persian Gulf, Strait of Hormuz transit interference, OPEC+ commentary on April production targets  
**Direction bias:** up  **Confidence:** medium  
**P(shock > $5) = 42%**  **Brier score:** 0.176

---
### 2026-02-23 — WTI $66.31/bbl  ·  🟢 Calm  (max 5-day move: $4.92, threshold $5)

<details><summary>📰 Oil market context (cutoff: 2026-02-22) — click to expand</summary>

### **Oil Market Intelligence Briefing**
**Date:** February 22, 2026  
**Subject:** Supply Risks and Geopolitical Volatility Outlook (5–10 Day Horizon)

#### **Price Action and Recent Trend**
As of late February 2026, WTI crude is trading in the **$72–$76 per barrel** range, showing a distinct bullish shift after a period of relative stability near $70 at the start of the month. Brent crude has seen a more pronounced move, currently carrying a **geopolitical risk premium of approximately $10/bbl** over its estimated fair value. This upward momentum is driven by a "short-squeeze" environment as market participants begin pricing in the possibility of a significant military escalation in the Middle East. While technical indicators show a +2.2% momentum gain over the last week, the market remains highly sensitive to headlines, with volatility expected to spike as we approach the end of the month.

#### **OPEC+ Policy and Supply Outlook**
The OPEC+ coalition is currently maintaining the production cuts reaffirmed in late 2025, keeping approximately **3.24 million barrels per day (mb/d)** off the market for Q1 2026. All eyes are now on the **March 1 Joint Ministerial Monitoring Committee (JMMC) meeting**. While the group’s official stance has been one of "precautionary stability," there are growing internal debates regarding the unwinding of voluntary cuts. Current intelligence suggests that while some members are eager to reclaim market share from non-OPEC producers (specifically the US and Guyana), the prevailing high-risk environment makes a sudden production increase unlikely in the next 10 days. Any signal from the March 1 meeting that cuts will be extended into Q2 would provide immediate further support to WTI prices.

#### **Geopolitical Risks: The Persian Gulf Powder Keg**
Geopolitical risk is at its highest level in years. Intelligence reports indicate a sharp increase in **anticipation of U.S. or Israeli military action against Iranian infrastructure**. While a formal conflict has not yet erupted, the "shadow war" has intensified, with markets pricing in a rising probability of a direct confrontation. The primary "tail risk" for the next 5–10 days is a threat to the **Strait of Hormuz**, a chokepoint for nearly 20% of global oil supply. Analysts warn that even a partial disruption or a credible threat of closure by Tehran would likely send WTI prices toward the **$85–$90** range almost instantly.

#### **Supply Disruption Signals and Sudden Move Factors**
Global oil inventories are currently assessed at a **tight 98–101 days of demand coverage**, leaving the market with a very thin buffer against physical shocks. In the next 5–10 days, a "sudden large move" (+$5–$10/bbl) in WTI could be triggered by:
1.  **Pre-emptive Strikes:** Any confirmed kinetic action against Iranian energy or nuclear sites.
2.  **Hormuz "Operational Pausing":** Reports of commercial tankers anchoring outside the Strait due to war-risk insurance spikes.
3.  

</details>

**Analyst Agent reasoning:**

> The market is entering a high-volatility window with the March 1 OPEC+ JMMC meeting coinciding with peak geopolitical tensions in the Persian Gulf. A $5 move is highly plausible given the reported 'short-squeeze' environment and the thin inventory buffer of 98-101 days. While no kinetic strike has occurred, the anticipation of military action against Iranian infrastructure creates a significant upside tail risk.

**Key signals:** March 1st OPEC+ JMMC meeting outcomes, Military escalation involving Iranian energy infrastructure, War-risk insurance spikes for Strait of Hormuz transit  
**Direction bias:** up  **Confidence:** medium  
**P(shock > $5) = 42%**  **Brier score:** 0.176

---
### 2026-03-02 — WTI $71.23/bbl  ·  🔴 SHOCK  (max 5-day move: $23.54, threshold $5)

<details><summary>📰 Oil market context (cutoff: 2026-03-01) — click to expand</summary>

**Oil Market Intelligence Briefing**
**Date:** March 1, 2026
**Subject:** Emergency Supply Risk Assessment – Persian Gulf Escalation

### **Market Snapshot: WTI/Brent Price Levels and Trends**
As of the morning of March 1, 2026, WTI crude is experiencing extreme "gap-up" volatility following the massive geopolitical shock of the last 24 hours. After closing February at approximately **$66.96/bbl**, WTI has surged in early trading, testing the **$80–$85** range as markets price in the immediate closure of the Strait of Hormuz. Brent crude, which ended February near **$71.32/bbl**, has spiked even more aggressively, with the Brent-WTI spread widening to over **$10/bbl** due to Brent’s higher exposure to the disrupted Middle Eastern waterborne trade. The steady upward trend observed throughout February—driven by rising regional tensions—has now transitioned into a vertical "fear rally."

### **Geopolitical Crisis: US-Israeli Strikes and Hormuz Closure**
The primary driver of the current price action is the large-scale, coordinated air campaign launched by US and Israeli forces against Iranian military and nuclear infrastructure on **February 28, 2026**. In immediate retaliation, the Iranian Revolutionary Guard Corps (IRGC) has effectively choked off the **Strait of Hormuz**, the world’s most critical oil chokepoint. Major shipping conglomerates, including Maersk and MSC, have officially suspended all transits through the Strait as of this morning. With approximately **20–21 million barrels per day (bpd)** of crude and refined products—roughly 20% of global consumption—now "trapped" within the Persian Gulf, the market is facing the largest single supply disruption in history.

### **OPEC+ Policy and Supply Outlook**
Prior to this weekend’s escalation, OPEC+ had reaffirmed a "stability" policy for Q1 2026, pausing previously planned production increases to manage a projected global surplus. However, that policy is now obsolete. With production from Saudi Arabia, Iraq, Kuwait, and the UAE (totaling over **14 million bpd**) unable to reach international markets via the Strait, the "supply glut" narrative has vanished. While Saudi Arabia and the UAE possess limited bypass pipelines to the Red Sea and Gulf of Oman, these can only handle roughly **3.5–5.5 million bpd**, leaving a massive net deficit. Non-OPEC+ producers, led by the "Americas Quintet" (US, Canada, Brazil, Guyana, and Argentina), are maximizing output, but cannot bridge a 15+ million bpd gap in the short term.

### **Risk Triggers for Sudden Large Moves (Next 5–10 Days)**
The following factors could cause WTI to move by **$10–$20/bbl** in either direction over the next 10 days:
*   **Retaliatory Strikes on Infrastructure:** Any IRGC drone or missile attacks on Saudi Arabia’s **Abqaiq** processing facility or the UAE’s **Fujairah** export terminals would remove the only remaining "bypass" options, potentially sending WTI toward **$120+**.
*   **Emergency SPR Release:** Rumors are circulating of a massive, coordinated Strategic Petroleum Reserve (SPR) release by IEA member nations (potentially up to **400 million barrels**). An official announcement would likely trigger a sharp, temporary "cool-off" in WTI prices.
*   **Escalation to Kharg Island:** If US/Israeli strikes expand to target Iran’s primary export terminal at **Kharg Island**, the market will price in a permanent loss of Iranian barrels, cementing a high-price floor.
*   **Insurance and Tanker Availability:** The withdrawal of war-risk insurance for the entire Middle East Gulf region could lead to a total cessation of even "shadow fleet" movements, further tightening the physical market.

</details>

**Analyst Agent reasoning:**

> The closure of the Strait of Hormuz is a catastrophic supply event, removing 20% of global oil flow and triggering immediate panic buying. Extreme volatility is guaranteed as the market weighs the loss of 20 million bpd against potential emergency SPR releases and further infrastructure strikes.

**Key signals:** Strait of Hormuz closure, US-Israeli military action, IEA emergency response  
**Direction bias:** up  **Confidence:** high  
**P(shock > $5) = 95%**  **Brier score:** 0.003

---
### 2026-03-09 — WTI $94.77/bbl  ·  🔴 SHOCK  (max 5-day move: $11.32, threshold $5)

<details><summary>📰 Oil market context (cutoff: 2026-03-08) — click to expand</summary>

### **Oil Market Intelligence Briefing**
**Date:** March 8, 2026  
**Subject:** Supply Risks and Geopolitical Escalation in the Persian Gulf

#### **Price Action and Market Sentiment**
As of March 8, 2026, WTI crude is trading near **$85.50/bbl**, while Brent has surged past the **$94.00** mark. The market is currently pricing in a massive "war premium" following the military escalations of late February. Prices spiked over 10% in the first week of March as traders reacted to the effective closure of the Strait of Hormuz. While the initial panic has slightly cooled, the trend remains aggressively bullish as global inventory draws are projected to reach 8.5 million b/d if the current maritime blockade persists through the second quarter.

#### **Geopolitical Crisis: The Hormuz Blockade**
The primary driver of volatility is the fallout from the **February 28, 2026, US-Israeli strikes** on Iranian military infrastructure. In immediate retaliation, Iran has effectively choked commercial traffic through the **Strait of Hormuz**, the world’s most critical oil chokepoint. Current intelligence indicates that over **200 tankers and LNG carriers** are currently anchored outside the Strait, unable to secure war-risk insurance or safe passage. With approximately 20 million b/d of crude and refined products (20% of global supply) normally transiting this route, the physical market is facing its most severe dislocation since the 1970s. Furthermore, Houthi forces in Yemen have resumed high-intensity drone attacks in the Red Sea, successfully closing both major Middle Eastern maritime corridors simultaneously.

#### **OPEC+ Policy and Supply Outlook**
OPEC+ remains in a "wait-and-see" posture following their late-2025 decision to **pause all production increases** through Q1 2026. The group is currently holding roughly 3.6 million b/d off the market. While there is immense diplomatic pressure on Saudi Arabia and the UAE to utilize their spare capacity via the East-West and Habshan-Fujairah pipelines, these land-based routes can only bypass approximately 35% of the Strait’s normal volume. Internal OPEC+ friction is rising; the UAE’s recent departure from the formal quota system has complicated the group's ability to coordinate a unified emergency response. No extraordinary meeting has been called yet, with the next ministerial session not scheduled until June 7, 2026.

#### **5-10 Day Outlook: Triggers for a Sudden Move**
The market is highly susceptible to a **$10–$15 swing** in either direction over the next 10 days based on two binary catalysts:
1.  **Downside Risk (Sudden Drop):** Rumors of a "back-channel" US-Iran ceasefire deal or a limited "humanitarian corridor" for tankers could trigger a massive long-liquidation event, potentially sending WTI back toward the $75 level instantly.
2.  **Upside Risk (Sudden Spike):** Any Iranian retaliatory strike against regional energy infrastructure—specifically the **Abqaiq processing facility** in Saudi Arabia or desalination plants in the UAE—would likely send WTI toward **$110/bbl** as the crisis shifts from a maritime disruption to a permanent loss of production capacity. 

**Analyst Note:** Monitor UKMTO (UK Maritime Trade Operations) reports for any "safe passage" signals, as shipping lines like Maersk and MSC are currently issuing daily updates on emergency surcharges that are further inflating the landed cost of crude.

</details>

**Analyst Agent reasoning:**

> The effective closure of the Strait of Hormuz has introduced extreme volatility, with WTI already moving nearly $14 in the last two trading sessions. A $5 move represents only a ~5.3% change from current levels, which is highly likely given the binary nature of the current geopolitical triggers (escalation vs. ceasefire). The physical market dislocation of 20 million b/d makes price stability nearly impossible in the short term.

**Key signals:** Strait of Hormuz transit status, Security of Saudi/UAE energy infrastructure, War-risk insurance availability for tankers  
**Direction bias:** up  **Confidence:** high  
**P(shock > $5) = 88%**  **Brier score:** 0.014

---
### 2026-03-16 — WTI $93.50/bbl  ·  🔴 SHOCK  (max 5-day move: $5.37, threshold $5)

<details><summary>📰 Oil market context (cutoff: 2026-03-15) — click to expand</summary>

**Oil Market Intelligence Briefing: March 15, 2026**

### **Price Action and Recent Trend**
WTI crude is currently trading in the **$92–$95 per barrel** range, marking a vertical ascent of over 20% since late February. Brent has already breached the psychological **$100 threshold**, trading near $104 as of this morning. The market is characterized by extreme "super-backwardation," indicating a desperate scramble for immediate physical barrels. While WTI remains at a historic discount to Brent (a spread widening toward $10–$12) due to high U.S. domestic inventories and the insulation of the Cushing hub, the domestic benchmark is under intense upward pressure as international buyers pivot toward North American grades to replace lost Middle Eastern supply.

### **OPEC+ Policy and Supply Disruptions**
The supply landscape has shifted from a projected 2026 surplus to the **largest sudden deficit in history**. Following the military escalation on February 28, 2026, approximately **10.5 to 13 million barrels per day (bpd)** of production from Saudi Arabia, the UAE, Kuwait, and Iraq has been effectively shut in or stranded. While OPEC+ had previously reaffirmed a production "pause" for Q1 2026 during their February 1 ministerial meeting, the alliance is now in crisis management mode. Saudi Arabia is attempting to reroute volumes through the East-West Pipeline to the Red Sea, but these efforts are hampered by the resumption of regional hostilities that have compromised alternative export terminals.

### **Geopolitical Risks and Shipping Lanes**
The "dual blockade" of the **Strait of Hormuz** and the **Bab el

</details>

**Analyst Agent reasoning:**

> The market is currently experiencing extreme volatility with daily price swings already exceeding $5/bbl as it processes a historic supply deficit of 10-13 million bpd. Given the active dual blockade of the Strait of Hormuz and Bab el-Mandeb, the scramble for physical barrels is likely to drive further high-amplitude moves as the WTI-Brent spread remains wide.

**Key signals:** 10-13 million bpd production shut-in, Strait of Hormuz and Bab el-Mandeb blockades, Extreme backwardation and widening Brent-WTI spread  
**Direction bias:** up  **Confidence:** high  
**P(shock > $5) = 85%**  **Brier score:** 0.023

---
### 2026-03-23 — WTI $88.13/bbl  ·  🔴 SHOCK  (max 5-day move: $14.75, threshold $5)

<details><summary>📰 Oil market context (cutoff: 2026-03-22) — click to expand</summary>

**Oil Market Intelligence Briefing**
**Date:** March 22, 2026
**Subject:** Supply Risks and Price Volatility Outlook (5–10 Day Horizon)

### **Price Level and Recent Trend**
As of March 22, 2026, WTI crude is trading in the **$94.00–$96.50** range, marking a dramatic 25% surge since the beginning of the month. This bullish breakout follows a period of relative stability in late 2025. The market is currently pricing in a significant "war premium" as spot prices have decoupled from long-term fundamentals. Brent has similarly breached the $100 threshold, driven by the near-total cessation of commercial traffic through the Strait of Hormuz. Technical indicators suggest the market is overbought, but the fundamental supply vacuum is preventing any meaningful correction.

### **OPEC+ Policy and Supply Outlook**
The OPEC+ alliance is currently in a state of functional paralysis. While the group officially reaffirmed a "production pause" for Q1 2026 during their November 2025 ministerial meeting, these quotas have been rendered moot by massive involuntary outages. Recent intelligence suggests that Middle Eastern members have seen a combined production drop of over **10 million barrels per day (bpd)** due to the escalating conflict. Furthermore, the market is reeling from the **UAE’s recent announcement that it will formally withdraw from OPEC effective May 1, 2026**, a move that has signaled a breakdown in group cohesion and raised fears of a future "market share war" once the current crisis abates.

### **Geopolitical Risks: The "Hormuz Blockade"**
The primary driver of the current

</details>

**Analyst Agent reasoning:**

> The market is currently in a state of extreme volatility, with WTI experiencing multiple swings exceeding $10/bbl within single trading sessions over the past two weeks. The combination of a 10 million bpd supply outage due to the Hormuz blockade and the structural shock of the UAE exiting OPEC creates a high-variance environment where price stability is non-existent. While the underlying supply vacuum is bullish, the recent $10 drop from $98 to $88 indicates that technical corrections and liquidation events are just as likely to trigger $5+ moves as geopolitical escalations.

**Key signals:** Strait of Hormuz commercial traffic cessation, 10 million bpd involuntary OPEC+ production outage, UAE formal withdrawal from OPEC  
**Direction bias:** neutral  **Confidence:** high  
**P(shock > $5) = 85%**  **Brier score:** 0.023

In [24]:
# ── Act 6 summary: per-week P(shock) estimates + cumulative Brier score ───────
#
# Top panel  — grouped bars, one cluster per origin: P(shock) for each method
#              with a small outcome marker (▲ calm / ▼ shock) above each group
# Bottom panel — cumulative mean Brier score over 8 weekly origins
#                annotated with a pre-shock / post-conflict background

METHOD_COLORS = {
    "Prophet":       CLR_PROPHET,
    "Analyst Agent": CLR_LLMP_CTX,
    "Always 50%":    "#aaa",
}
METHOD_DASH = {
    "Prophet":       "solid",
    "Analyst Agent": "solid",
    "Always 50%":    "dot",
}

origins_ordered  = [o.strftime("%Y-%m-%d") for o in SHOCK_ORIGINS]
outcome_by_key   = {
    key: int(binary_df[(binary_df["origin"] == key) & (binary_df["method"] == "Prophet")].iloc[0]["outcome"])
    for key in origins_ordered
}

fig = psp.make_subplots(
    rows=2, cols=1,
    row_heights=[0.50, 0.50],
    vertical_spacing=0.14,
    subplot_titles=[
        "P(WTI closes >+$5/bbl in 5 days) — Prophet · Analyst Agent · Always-50%",
        "Cumulative mean Brier score over 8 weeks (lower = better, 0.25 = random ceiling)",
    ],
)

# ── Row 1: grouped bars — P(shock) per origin per method ─────────────────────
for method in ["Analyst Agent", "Prophet", "Always 50%"]:
    sub = binary_df[binary_df["method"] == method].sort_values("origin_dt")
    fig.add_trace(go.Bar(
        x=sub["origin"],
        y=sub["prob"],
        name=method,
        marker_color=METHOD_COLORS[method],
        marker_opacity=0.82,
        legendgroup=method,
        showlegend=True,
        text=[f"{p:.0%}" for p in sub["prob"]],
        textposition="outside",
        textfont=dict(size=8),
    ), row=1, col=1)

# Outcome annotations above each origin cluster
for key in origins_ordered:
    outcome = outcome_by_key[key]
    icon    = "SHOCK" if outcome else "calm"
    colour  = CLR_CONFLICT if outcome else "#2ca02c"
    fig.add_annotation(
        x=key, y=1.12, text=f"{'▼' if outcome else '▲'} {icon}",
        showarrow=False, font=dict(size=9, color=colour),
        xref="x", yref="y",
    )

fig.update_yaxes(title_text="P(up > +$5/bbl)", range=[0, 1.20],
                 tickformat=".0%", row=1, col=1)
fig.update_xaxes(tickangle=-30, row=1, col=1)

# Pre-shock / post-conflict shading (row 1)
fig.add_vrect(
    x0="2026-02-02", x1="2026-03-01",
    fillcolor="rgba(33,113,181,0.04)", line_width=0,
    annotation_text="pre-conflict", annotation_position="top left",
    annotation_font=dict(size=9, color="#2171b5"),
    row=1, col=1,
)
fig.add_vrect(
    x0="2026-03-01", x1="2026-03-23",
    fillcolor="rgba(214,39,40,0.04)", line_width=0,
    annotation_text="post-conflict onset", annotation_position="top right",
    annotation_font=dict(size=9, color="#d62728"),
    row=1, col=1,
)

# ── Row 2: cumulative mean Brier score over 8 origins ─────────────────────────
for method in ["Analyst Agent", "Prophet", "Always 50%"]:
    sub       = binary_df[binary_df["method"] == method].sort_values("origin_dt")
    cum_brier = sub["brier"].expanding().mean().values
    fig.add_trace(go.Scatter(
        x=sub["origin"].values,
        y=cum_brier,
        name=method,
        mode="lines+markers",
        line=dict(color=METHOD_COLORS[method], dash=METHOD_DASH[method], width=2.5),
        marker=dict(size=8),
        legendgroup=method,
        showlegend=False,
        hovertemplate="%{x}<br>Cumulative Brier: %{y:.3f}<extra>" + method + "</extra>",
    ), row=2, col=1)

# Reference ceiling at 0.25 (always-50% theoretical Brier)
fig.add_hline(
    y=0.25, line=dict(color="#aaa", dash="dot", width=1.5),
    annotation_text="0.25 — always-50% ceiling",
    annotation_position="top left",
    annotation_font=dict(size=9, color="#888"),
    row=2, col=1,
)

# Shading (row 2)
fig.add_vrect(
    x0="2026-02-02", x1="2026-03-01",
    fillcolor="rgba(33,113,181,0.04)", line_width=0,
    row=2, col=1,
)
fig.add_vrect(
    x0="2026-03-01", x1="2026-03-23",
    fillcolor="rgba(214,39,40,0.04)", line_width=0,
    row=2, col=1,
)

fig.update_yaxes(
    title_text="Cumulative mean Brier score", range=[0, 0.34],
    row=2, col=1,
)
fig.update_xaxes(tickangle=-30, row=2, col=1)

# Per-origin Brier dots on the cumulative line (small circle markers)
for method in ["Analyst Agent", "Prophet"]:
    sub       = binary_df[binary_df["method"] == method].sort_values("origin_dt")
    for _, row_data in sub.iterrows():
        # Running mean up to this row
        idx = list(sub["origin"]).index(row_data["origin"])
        running_mean = sub["brier"].iloc[:idx + 1].mean()
        fig.add_annotation(
            x=row_data["origin"],
            y=running_mean,
            text=f"{running_mean:.3f}",
            showarrow=False,
            font=dict(size=7, color=METHOD_COLORS[method]),
            yshift=9,
            xref="x2", yref="y2",
        )

fig.update_layout(
    title=dict(
        text="Upward Price Shock Forecasting — Feb–Mar 2026: Analyst Agent vs. Prophet",
        x=0.5, font=dict(size=14),
    ),
    height=620,
    barmode="group",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1, font=dict(size=11)),
    margin=dict(t=80, b=60, l=70, r=30),
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor="#ececec", gridwidth=0.7)
fig.show()

---

## Act 5 — Trajectory Evaluation
what would a proper forecasting agent do with more than a text snippet?

In [25]:
pass  # act7 causal analysis removed — see cumulative Brier chart in Act 6rigin) & (llmp_df["horizon"] == HORIZON_B)]
    bare = sub[sub["context_tag"] == "bare"]
    ctx  = sub[sub["context_tag"] == "context"]
    bare_med = float(bare["median"].iloc[0]) if not bare.empty else float("nan")
    ctx_med  = float(ctx["median"].iloc[0])  if not ctx.empty  else float("nan")
    _, actual = resolution_price(price_df, origin)
    origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])

    shift_rows.append({
        "Origin":             ORIGIN_CONTEXTS[key]["label"],
        "Actual price":       f"${actual:.0f}",
        "LLMP bare median":   f"${bare_med:.1f}",
        "LLMP+ctx median":    f"${ctx_med:.1f}",
        "Context shift":      f"+${ctx_med - bare_med:.1f}" if ctx_med >= bare_med else f"${ctx_med - bare_med:.1f}",
        "Actual vs. bare":    f"{actual - bare_med:+.1f}",
        "Actual vs. +ctx":    f"{actual - ctx_med:+.1f}",
        "Context in context": ORIGIN_CONTEXTS[key]["context_text"].split("\n")[0],
    })

shift_df = pd.DataFrame(shift_rows)
display_cols = ["Origin", "LLMP bare median", "LLMP+ctx median", "Context shift", "Actual price", "Actual vs. bare", "Actual vs. +ctx"]
shift_df[display_cols]

,Origin,LLMP bare median,LLMP+ctx median,Context shift,Actual price,Actual vs. bare,Actual vs. +ctx
0,"Jan 5, 2026",$61.0,$58.0,$-2.9,$65,+4.2,+7.1
1,"Feb 2, 2026",$61.8,$78.5,+$16.8,$75,+12.9,-3.9
2,"Mar 2, 2026",$68.4,$96.3,+$27.9,$100,+31.7,+3.8


Point MAE at the 30-day horizon and directional accuracy across the three trajectory origins.e / Actual vs. +ctx** is the residual error. Smaller (closer to 0) is better.
- The March origin is the most telling: even a large positive context shift still
  undershoots the actual $100 price, because no purely-text-based model can fully
  price in a live supply disruption without numerical data to anchor on.

### This is the case for a Track 2 Analyst Agent

The LLMP with a text snippet is the *minimum viable context-aware forecaster*:
it can move in the right direction, but it works from headlines, not data.

| What we did here | What a Track 2 Analyst Agent would do |
|---|---|
| Manually curated a text snippet | Retrieves context automatically — news, futures, sentiment |
| Context is unverified prose | Queries DataService; validates signals in code |
| Single point-of-view forecast | Runs explicit scenarios ("blockade persists 60 days" vs. "resolves in 2 weeks") |
| No explanation | Produces a written narrative: sources, assumptions, confidence level |
| No follow-up | Conversational — answers "what would change your forecast?" |

> *"The LLMP with context is already seeing something Prophet cannot.
> The Analyst Agent is the version that knows what to look for and explains why."*

In [26]:
eval_rows = []

for origin in ORIGINS:
    key = origin.strftime("%Y-%m-%d")
    _, actual = resolution_price(price_df, origin)
    origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])
    true_direction = actual > origin_price

    # Prophet
    p_row = prophet_row_at_origin(prophet_df, origin)
    p_mae = abs(p_row["yhat"] - actual)
    eval_rows.append({
        "Origin":     ORIGIN_CONTEXTS[key]["label"],
        "Method":     "Prophet",
        "Forecast":   f"${p_row['yhat']:.1f}",
        "Actual":     f"${actual:.1f}",
        "MAE ($)": f"{p_mae:.1f}",
        "Dir. correct": "✓" if (p_row["yhat"] > origin_price) == true_direction else "✗",
        "Inside CI": str(p_row["inside_ci"]),
    })

    # LLMP
    for tag in ["bare", "context"]:
        sub = llmp_df[(llmp_df["origin"] == origin) & (llmp_df["context_tag"] == tag) & (llmp_df["horizon"] == HORIZON_B)]
        if sub.empty:
            continue
        med = float(sub.iloc[0]["median"])
        mae = abs(med - actual)
        eval_rows.append({
            "Origin":     ORIGIN_CONTEXTS[key]["label"],
            "Method":     f"LLMP — {tag}",
            "Forecast":   f"${med:.1f}",
            "Actual":     f"${actual:.1f}",
            "MAE ($)": f"{mae:.1f}",
            "Dir. correct": "✓" if (med > origin_price) == true_direction else "✗",
            "Inside CI": "—",
        })

eval_df = pd.DataFrame(eval_rows)
eval_df

,Origin,Method,Forecast,Actual,MAE ($),Dir. correct,Inside CI
0,"Jan 5, 2026",Prophet,$57.5,$65.1,7.6,✗,True
1,"Jan 5, 2026",LLMP — bare,$61.0,$65.1,4.2,✓,—
2,"Jan 5, 2026",LLMP — context,$58.0,$65.1,7.1,✗,—
3,"Feb 2, 2026",Prophet,$60.9,$74.7,13.7,✗,False
4,"Feb 2, 2026",LLMP — bare,$61.8,$74.7,12.9,✗,—
5,"Feb 2, 2026",LLMP — context,$78.5,$74.7,3.9,✓,—
6,"Mar 2, 2026",Prophet,$61.3,$100.1,38.8,✗,False
7,"Mar 2, 2026",LLMP — bare,$68.4,$100.1,31.7,✗,—
8,"Mar 2, 2026",LLMP — context,$96.3,$100.1,3.8,✓,—


### Act 6 — Evaluation: Brier score summary

**Question:** P(WTI closes > today + $5/bbl in 5 trading days)  
**Brier score** = (P − outcome)².  Lower is better.  **0.25 = always-50% ceiling.**

In [27]:
# ── Per-origin Brier table ─────────────────────────────────────────────────────
pivot = (
    binary_df
    .assign(
        prob_str=lambda d: d["prob"].map(lambda p: f"{p:.0%}"),
        brier_str=lambda d: d["brier"].map(lambda b: f"{b:.3f}"),
        summary=lambda d: d["prob_str"] + "  (BS " + d["brier_str"] + ")",
    )
    .pivot_table(index="origin", columns="method", values="summary", aggfunc="first")
)
pivot.index.name = "Origin"
pivot.columns.name = None

meta = binary_df.groupby("origin").first()[["outcome", "max_move"]].reindex(pivot.index)
meta.columns = ["Upward shock? (1=yes)", "5-day Δ ($)"]
meta["5-day Δ ($)"] = meta["max_move"].map("{:+.2f}".format)

col_order = ["Upward shock? (1=yes)", "5-day Δ ($)"] + [
    c for c in ["Always 50%", "Prophet", "Analyst Agent"] if c in pivot.columns
]
result = pd.concat([meta, pivot], axis=1)[col_order]
display(result)

# ── Mean Brier score per method ────────────────────────────────────────────────
mean_brier = (
    binary_df.groupby("method")["brier"].mean()
    .reindex(["Always 50%", "Prophet", "Analyst Agent"])
    .rename("Mean Brier score")
    .map("{:.4f}".format)
    .to_frame()
)
mean_brier.index.name = "Method"
print("\nOverall mean Brier score (lower = better, 0.25 = random ceiling):")
display(mean_brier)

,Shock? (1=yes),Max |Δ| ($),Always 50%,Prophet,Analyst Agent
Origin,,,,,
2026-02-02,0,3.00,50% (BS 0.250),38% (BS 0.141),42% (BS 0.176)
2026-02-09,0,2.03,50% (BS 0.250),47% (BS 0.216),22% (BS 0.048)
2026-02-17,0,4.10,50% (BS 0.250),23% (BS 0.054),42% (BS 0.176)
2026-02-23,0,4.92,50% (BS 0.250),38% (BS 0.144),42% (BS 0.176)
2026-03-02,1,23.54,50% (BS 0.250),73% (BS 0.075),95% (BS 0.003)
2026-03-09,1,11.32,50% (BS 0.250),100% (BS 0.000),88% (BS 0.014)
2026-03-16,1,5.37,50% (BS 0.250),100% (BS 0.000),85% (BS 0.023)
2026-03-23,1,14.75,50% (BS 0.250),100% (BS 0.000),85% (BS 0.023)



Overall mean Brier score (lower = better, 0.25 = random ceiling):


,Mean Brier score
Method,
Always 50%,0.2500
Prophet,0.0789
Analyst Agent,0.0799
